In [3]:
# --- Final Combined Preprocessing Pipeline ---
# This script integrates all member tasks into a final, logical workflow.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def load_and_select_data(file_path):
    """Loads the dataset and selects only the relevant columns for the project."""
    print("--- Step 0: Loading Data and Selecting Relevant Columns ---")
    relevant_cols = ['Area', 'Item', 'Year', 'Value']
    df = pd.read_csv(file_path, usecols=relevant_cols)
    print(f"Data loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.\n")
    return df

# --- STEP 1:Member 1- Handling Irrelevant Columns---
def handle_missing_values(df):
    """
    Member 1's Task: Fills missing 'Value' entries with the column median.
    Note: This step simulates missing data as the original dataset is complete.
    """
    print("--- Step 1 (Member 1): Handling Missing Values ---")
    df_processed = df.copy()
    np.random.seed(42)
    missing_indices = np.random.choice(df_processed.index, size=300, replace=False)
    df_processed.loc[missing_indices, 'Value'] = np.nan
    print(f"Simulated {df_processed['Value'].isnull().sum()} missing values to demonstrate imputation.")
    median_value = df_processed['Value'].median()
    df_processed['Value'] = df_processed['Value'].fillna(median_value)
    print(f"Filled missing values using the column median ({median_value:.2f}).\n")
    return df_processed

# --- STEP 2 & 3: Members 5 & 6 - Feature Engineering and Selection ---
def engineer_and_select_features(df):
    """Combines feature creation (Member 5) and selection (Member 6)."""
    print("--- Step 2 & 3 (Members 5 & 6): Feature Engineering & Selection ---")
    start_year = df['Year'].min()
    df_engineered = df.copy()
    df_engineered['Years_Since_Start'] = df_engineered['Year'] - start_year
    df_selected = df_engineered.drop(columns=['Year'])
    print("Created 'Years_Since_Start' and removed original 'Year' column.\n")
    return df_selected

# --- STEP 4: Member 2 - Encoding Categorical Variables ---
def encode_categorical_variables(df):
    """
    Member 2's Task: Converts text features into numerical format.
    ADJUSTED to ensure the final feature count is >= 6 AFTER the target is engineered.
    """
    print("--- Step 4 (Member 2): Encoding Categorical Variables ---")
    df_encoded = df.copy()
    df_encoded = pd.get_dummies(df_encoded, columns=['Item'], prefix='Item', drop_first=True)

    top_5_areas = df_encoded['Area'].value_counts().nlargest(5).index
    df_encoded['Area'] = df_encoded['Area'].where(df_encoded['Area'].isin(top_5_areas), 'Other')
    df_encoded = pd.get_dummies(df_encoded, columns=['Area'], prefix='Area', drop_first=True)
    
    print("'Item' and 'Area' columns have been converted to numerical format using One-Hot Encoding.\n")
    return df_encoded

# --- STEP 5: Member 3 - Removing Outliers ---
def remove_outliers(df):
    """Member 3's Task: Removes statistical outliers from the 'Value' column."""
    print("--- Step 5 (Member 3): Removing Outliers from 'Value' Column ---")
    df_no_outliers = df.copy()
    Q1 = df_no_outliers['Value'].quantile(0.25)
    Q3 = df_no_outliers['Value'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    initial_rows = len(df_no_outliers)
    df_no_outliers = df_no_outliers[df_no_outliers['Value'] <= upper_bound]
    removed_count = initial_rows - len(df_no_outliers)
    print(f"Removed {removed_count} outliers based on the IQR method.\n")
    return df_no_outliers

# --- STEP 6: Member 4 - Scaling Numerical Features ---
def scale_features(df):
    """Member 4's Task: Applies Min-Max scaling to all numerical features."""
    print("--- Step 6 (Member 4): Scaling All Numerical Features ---")
    df_scaled = df.copy()
    scaler = MinMaxScaler()
    numerical_cols = df_scaled.select_dtypes(include=np.number).columns
    df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])
    print("All numerical features have been scaled to a range of [0, 1].\n")
    return df_scaled

# --- NEW STEP: Convert Regression Task to Classification Task ---
def convert_to_classification(df):
    """
    Adds a final step to convert the regression problem into a binary
    classification problem by creating a categorical target variable.
    """
    print("--- Final Step: Converting Target for Classification ---")
    df_class = df.copy()
    median_value = df_class['Value'].median()
    print(f"Using median of scaled 'Value' ({median_value:.2f}) as the threshold.")
    df_class['Is_High_Area'] = (df_class['Value'] > median_value).astype(int)
    df_class = df_class.drop(columns=['Value'])
    print("Created new binary target 'Is_High_Area' and dropped the original 'Value' feature.\n")
    return df_class

In [4]:
# --- Main Execution Block ---
if __name__ == '__main__':
    FILE_PATH = 'FAOSTAT_data_en_8-15-2025.csv'
    
    df_loaded = load_and_select_data(FILE_PATH)
    df_step1 = handle_missing_values(df_loaded)
    df_step2 = engineer_and_select_features(df_step1)
    df_step3 = encode_categorical_variables(df_step2)
    df_step4 = remove_outliers(df_step3)
    df_scaled = scale_features(df_step4)
   
    # Call the new function to convert the target variable
    df_final = convert_to_classification(df_scaled)
    
    print("--- PIPELINE FOR CLASSIFICATION COMPLETE ---")
    # The final dataset now has 7 features and 1 target variable
    print(f"Final dataset has {df_final.shape[1] - 1} features and 1 binary target, meeting the requirement.")
    print("Final Preprocessed Data Head:")
    print(df_final.head())
    
    output_filename = 'preprocessed_classification_data.csv'
    df_final.to_csv(output_filename, index=False)
    print(f"\nFinal dataset for classification saved successfully to '{output_filename}'")

--- Step 0: Loading Data and Selecting Relevant Columns ---
Data loaded successfully with 4054 rows and 4 columns.

--- Step 1 (Member 1): Handling Missing Values ---
Simulated 300 missing values to demonstrate imputation.
Filled missing values using the column median (138.86).

--- Step 2 & 3 (Members 5 & 6): Feature Engineering & Selection ---
Created 'Years_Since_Start' and removed original 'Year' column.

--- Step 4 (Member 2): Encoding Categorical Variables ---
'Item' and 'Area' columns have been converted to numerical format using One-Hot Encoding.

--- Step 5 (Member 3): Removing Outliers from 'Value' Column ---
Removed 779 outliers based on the IQR method.

--- Step 6 (Member 4): Scaling All Numerical Features ---
All numerical features have been scaled to a range of [0, 1].

--- Final Step: Converting Target for Classification ---
Using median of scaled 'Value' (0.00) as the threshold.
Created new binary target 'Is_High_Area' and dropped the original 'Value' feature.

--- PIPE